In [ ]:
import os

import pandas as pd

from pynxtools_em.examples.oasisb.oasisb_utils import (
    CSV_HEADER_FOR_HASH_FILE,
    get_project_id,
)

print(os.getcwd())
with open("source_directory.txt") as fp:
    src_directory = f"{fp.readline().strip().replace('/', os.sep)}"
print(src_directory)

In [ ]:
spread_sheet_of_all_projects = pd.read_excel(
    f"{src_directory}{os.sep}aaa_legacy_data.ods",
    sheet_name="aaa_legacy_data",
    engine="odf",
    dtype=str,
).fillna("")

In [ ]:
index: dict[str, dict[str, list[str]]] = {}

In [ ]:
def populate_index(project_id: str, index) -> index:
    """Read f'{project_id}.sha256.results.csv' file, modify idx project_id as first-level key, use SHA256 as second-level key, list[str] stores all files with that hash."""
    if project_id in index:
        print(f"WARNING::{project_id} already exists, do not overwrite")
        return index
    checksum_file_name = f"{project_id}.sha256.results.csv"
    if os.path.isfile(f"{src_directory}{os.sep}{checksum_file_name}"):
        with open(f"{src_directory}{os.sep}{checksum_file_name}") as fp:
            txt = fp.readlines()
            if not txt[6].startswith(CSV_HEADER_FOR_HASH_FILE):
                print(f"WARNING::{checksum_file_name} formatted incorrectly")

            index[project_id] = {}
            for idx, line in enumerate(txt[7:]):
                tmp = [token.strip() for token in line.strip().split(";")]
                if len(tmp) == 4:
                    if not tmp[0].endswith("/"):  # do not add paths to directories
                        path = tmp[0]
                        sha = tmp[3]
                        if sha not in index[project_id]:
                            index[project_id][sha] = [path]
                        else:
                            index[project_id][sha].append(path)
                # else:
                #     print(tmp)
    else:
        print(f"WARNING::{project_id} not included")
    return index

In [ ]:
# index = populate_index("000", index)
for row in spread_sheet_of_all_projects.itertuples(index=True):
    project_id = get_project_id(row.project_name)
    if os.path.isfile(f"{src_directory}{os.sep}{project_id}.sha256.results.csv"):
        index = populate_index(project_id, index)
        print(f"{project_id}, {len(index[project_id])}")
print("Index build successfully")

In [ ]:
# search if for a project in the scratch directory f"{src_directory}{os.sep}000 we have all its files in a directory of another upload
def search_for_project_data(prefix: str):
    checksum_file_name = "000.sha256.results.csv"
    with open(f"{src_directory}{os.sep}{checksum_file_name}") as fp:
        txt = fp.readlines()
        if not txt[6].startswith(CSV_HEADER_FOR_HASH_FILE):
            print(f"WARNING::{checksum_file_name} formatted incorrectly")

        asked = 0
        found = 0
        projects = set()
        for idx, line in enumerate(txt[7:]):
            tmp = [token.strip() for token in line.strip().split(";")]
            if len(tmp) == 4:
                if not tmp[0].endswith("/"):  # do not add paths to directories
                    if prefix in tmp[0]:
                        asked += 1
                        sha = tmp[3]
                        for key, lookup in index.items():
                            if sha in lookup:
                                found += 1
                                projects.add(key)
                                # print(f"{sha} found in project_id {key}")  # , {index[key][sha]}")
                                break
        print(f"found / asked >>>> {found} / {asked}, {projects}")

In [ ]:
file_paths: list[str] = []

for file_path in sorted(file_paths):
    print(file_path)
    search_for_project_data(prefix=file_path)

In [ ]:
query = [
    token.strip()
    for token in "10.25430researchdata.cab.unipd.it.00000898.zip:19-37_Site_1.ctf;64776572;1718198402.0;744d170d9354a672ab94f2e4d8345c526a5a3ddfe6a7fcc79aa9b1c8e241d044".split(
        ";"
    )
]
print(query)
# is there an artifact with that same SHA256 ?

In [ ]:
for idx, key in enumerate(index["000"]):
    if len(index["000"][key]) != 1:
        print(key)
        break
        print(index["000"][key])
        if idx > 10:
            break

In [ ]:
print(index["000"]["a3a63108e9355a6988bcef27e584740f234e56fc5c2c6a17b0da1617664077c0"])